In [1]:
# Importing libraries & frameworks 
import random
import genesis as gs
import numpy as np
import torch
import torch.nn.functional as F
import cv2
import json
from pathlib import Path
from scipy.spatial.transform import Rotation as R

[I 04/23/25 16:44:46.878 676014] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


In [2]:
# from pytorch3d

def _axis_angle_rotation(axis: str, angle: torch.Tensor) -> torch.Tensor:
    """
    Return the rotation matrices for one of the rotations about an axis
    of which Euler angles describe, for each value of the angle given.

    Args:
        axis: Axis label "X" or "Y or "Z".
        angle: any shape tensor of Euler angles in radians

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """

    cos = torch.cos(angle)
    sin = torch.sin(angle)
    one = torch.ones_like(angle)
    zero = torch.zeros_like(angle)

    if axis == "X":
        R_flat = (one, zero, zero, zero, cos, -sin, zero, sin, cos)
    elif axis == "Y":
        R_flat = (cos, zero, sin, zero, one, zero, -sin, zero, cos)
    elif axis == "Z":
        R_flat = (cos, -sin, zero, sin, cos, zero, zero, zero, one)
    else:
        raise ValueError("letter must be either X, Y or Z.")

    return torch.stack(R_flat, -1).reshape(angle.shape + (3, 3))

def euler_angles_to_matrix(euler_angles: torch.Tensor, convention: str) -> torch.Tensor:
    """
    Convert rotations given as Euler angles in radians to rotation matrices.

    Args:
        euler_angles: Euler angles in radians as tensor of shape (..., 3).
        convention: Convention string of three uppercase letters from
            {"X", "Y", and "Z"}.

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    if euler_angles.dim() == 0 or euler_angles.shape[-1] != 3:
        raise ValueError("Invalid input euler angles.")
    if len(convention) != 3:
        raise ValueError("Convention must have 3 letters.")
    if convention[1] in (convention[0], convention[2]):
        raise ValueError(f"Invalid convention {convention}.")
    for letter in convention:
        if letter not in ("X", "Y", "Z"):
            raise ValueError(f"Invalid letter {letter} in convention string.")
    matrices = [
        _axis_angle_rotation(c, e)
        for c, e in zip(convention, torch.unbind(euler_angles, -1))
    ]
    # return functools.reduce(torch.matmul, matrices)
    return torch.matmul(torch.matmul(matrices[0], matrices[1]), matrices[2])

def standardize_quaternion(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert a unit quaternion to a standard form: one in which the real
    part is non negative.

    Args:
        quaternions: Quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Standardized quaternions as tensor of shape (..., 4).
    """
    return torch.where(quaternions[..., 0:1] < 0, -quaternions, quaternions)

def quaternion_raw_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Multiply two quaternions.
    Usual torch rules for broadcasting apply.

    Args:
        a: Quaternions as tensor of shape (..., 4), real part first.
        b: Quaternions as tensor of shape (..., 4), real part first.

    Returns:
        The product of a and b, a tensor of quaternions shape (..., 4).
    """
    aw, ax, ay, az = torch.unbind(a, -1)
    bw, bx, by, bz = torch.unbind(b, -1)
    ow = aw * bw - ax * bx - ay * by - az * bz
    ox = aw * bx + ax * bw + ay * bz - az * by
    oy = aw * by - ax * bz + ay * bw + az * bx
    oz = aw * bz + ax * by - ay * bx + az * bw
    return torch.stack((ow, ox, oy, oz), -1)

def quaternion_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Multiply two quaternions representing rotations, returning the quaternion
    representing their composition, i.e. the versor with nonnegative real part.
    Usual torch rules for broadcasting apply.

    Args:
        a: Quaternions as tensor of shape (..., 4), real part first.
        b: Quaternions as tensor of shape (..., 4), real part first.

    Returns:
        The product of a and b, a tensor of quaternions of shape (..., 4).
    """
    ab = quaternion_raw_multiply(a, b)
    return standardize_quaternion(ab)

def _sqrt_positive_part(x: torch.Tensor) -> torch.Tensor:
    """
    Returns torch.sqrt(torch.max(0, x))
    but with a zero subgradient where x is 0.
    """
    ret = torch.zeros_like(x)
    positive_mask = x > 0
    if torch.is_grad_enabled():
        ret[positive_mask] = torch.sqrt(x[positive_mask])
    else:
        ret = torch.where(positive_mask, torch.sqrt(x), ret)
    return ret

def matrix_to_quaternion(matrix: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to quaternions.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    """
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")

    batch_dim = matrix.shape[:-2]
    m00, m01, m02, m10, m11, m12, m20, m21, m22 = torch.unbind(
        matrix.reshape(batch_dim + (9,)), dim=-1
    )

    q_abs = _sqrt_positive_part(
        torch.stack(
            [
                1.0 + m00 + m11 + m22,
                1.0 + m00 - m11 - m22,
                1.0 - m00 + m11 - m22,
                1.0 - m00 - m11 + m22,
            ],
            dim=-1,
        )
    )

    # we produce the desired quaternion multiplied by each of r, i, j, k
    quat_by_rijk = torch.stack(
        [
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([q_abs[..., 0] ** 2, m21 - m12, m02 - m20, m10 - m01], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m21 - m12, q_abs[..., 1] ** 2, m10 + m01, m02 + m20], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m02 - m20, m10 + m01, q_abs[..., 2] ** 2, m12 + m21], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m10 - m01, m20 + m02, m21 + m12, q_abs[..., 3] ** 2], dim=-1),
        ],
        dim=-2,
    )

    # We floor here at 0.1 but the exact level is not important; if q_abs is small,
    # the candidate won't be picked.
    flr = torch.tensor(0.1).to(dtype=q_abs.dtype, device=q_abs.device)
    quat_candidates = quat_by_rijk / (2.0 * q_abs[..., None].max(flr))

    # if not for numerical problems, quat_candidates[i] should be same (up to a sign),
    # forall i; we pick the best-conditioned one (with the largest denominator)
    out = quat_candidates[
        F.one_hot(q_abs.argmax(dim=-1), num_classes=4) > 0.5, :
    ].reshape(batch_dim + (4,))
    return standardize_quaternion(out)



In [ ]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    precision='32',
    debug=False,
    eps=1e-12,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    sim_options=gs.options.SimOptions(dt=0.001),
    show_viewer=True, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())

bottle_radius = 0.025
bottle_pos = [0, 0.0, 0.1] 

plastic_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="../bottles/plastic_bottle/bottle.stl", 
        pos=bottle_pos,            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.0002                      
    )
)
initial_gripper_pos = [-0.5, 0.0, 0.10]
initial_gripper_quat = [0.707, 0.707, 0, 0]
spot_gripper = scene.add_entity(
    gs.morphs.URDF(
        file='/home/nexus/Desktop/spot_gen/spot_arm_description/spot_arm/urdf/open_gripper.urdf',
        quat = initial_gripper_quat,
        pos=initial_gripper_pos,
        scale=1.0,
        merge_fixed_links=True,
        fixed = True
    ),
)
# cylinder_radius = 0.01
# cylinder_pos = [0.5, 0.0, 0.05] 
# spot_gripper = scene.add_entity(
#     gs.morphs.Cylinder(
#         pos=cylinder_pos,
#         height=0.01,
#         radius=cylinder_radius,
#         collision=True,
#     )
# )

# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = (0.0, 0.0, 0.09),
    res    = (640, 480),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)
# Building the Scene
scene.build(n_envs=5)

[Genesis] [16:44:49] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [16:44:49] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [16:44:49] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [16:44:49] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [16:44:49] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [16:44:50] [INFO] Scene <c1c1a46> created.
[Genesis] [16:44:50] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <c51ca72>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [16:44:50] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <e6d18fc>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/spot_gen/Exp_RigidEntity/bottles/plastic_bottle/bottle.stl')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [16:44:51] [INFO] Adding <gs.RigidEntity>. idx: 2, uid: <2c7b007>, morph: <gs.morphs.URDF(file='/home/nexus/Desktop/spot_gen/spot_arm_description/spot_arm/urdf/open_gripper.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [16:44:51] [INFO] Building scene <c1c1a46>...
[Genesis] [16:44:58] [INFO] Compiling simulation kernels...
[Genesis] [16:45:25] [INFO] Building visualizer...
[Genesis] [16:45:26] [INFO] Viewer created. Resolution: 667×500, max_FPS: 600.


/home/nexus/miniconda3/lib/python3.12/site-packages/genesis/ext/pyrender/viewer.py


In [4]:
# Define DOFs
hand_dof = np.arange(2)
finger_dof = np.array([1]) 

# Set PD control parameters
spot_gripper.set_dofs_kp(
    np.array([100]*2)
    )
spot_gripper.set_dofs_kv(
    np.array([1]*2)
    )
spot_gripper.set_dofs_force_range(
    np.array([-100]*2),
    np.array([100]*2)
    )

In [5]:
def axis_angle_to_quaternion(axis_angle: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as axis/angle to quaternions.

    Args:
        axis_angle: Rotations given as a vector in axis angle form,
            as a tensor of shape (..., 3), where the magnitude is
            the angle turned anticlockwise in radians around the
            vector's direction.

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    """
    angles = torch.norm(axis_angle, p=2, dim=-1, keepdim=True)
    sin_half_angles_over_angles = 0.5 * torch.sinc(angles * 0.5 / torch.pi)
    return torch.cat(
        [torch.cos(angles * 0.5), axis_angle * sin_half_angles_over_angles], dim=-1
    )

def vectors_to_quaternion(vectors):
    """
    Convert a batch of 3D vectors to Quaternion angles (wxyz convention).
    
    Parameters:
    vectors : np.ndarray, shape (n, 3)
        Array of n 3D vectors.
    
    Returns:
    quaternions : np.ndarray, shape (n, 4)
        Array of quaternions for each input vector.
    """
    # Ensure input is a PyTorch tensor with shape (n, 3)
    vectors = torch.as_tensor(vectors, dtype=torch.float32)
    if vectors.ndim != 2 or vectors.shape[1] != 3:
        raise ValueError("Input must be a tensor of shape (n, 3)")
    
    # Normalize the vectors
    norms = torch.norm(vectors, dim=1, keepdim=True)  # (n, 1)
    if torch.any(norms == 0):
        raise ValueError("Input contains zero vectors")
    normalized_vectors = vectors / norms
    
    # Reference vector
    reference = torch.tensor([1.0, 0.0, 0.0])
    
    # Compute rotation axes and angles
    dots = torch.clamp(torch.sum(normalized_vectors * reference, dim=1), -1.0, 1.0)
    angles = torch.arccos(dots)
    
    axes = torch.cross(reference.expand_as(normalized_vectors), normalized_vectors)
    axis_norms = torch.norm(axes, dim=1, keepdim=True)
    axes = axes / (axis_norms + 1e-10)  # Avoid division by zero
        
    # Compute rotation vectors (axis-angle)
    rotvecs = angles[:, None] * axes
        
    # Convert to quaternions using provided function
    quaternions = axis_angle_to_quaternion(rotvecs)

    # # Handle cases where vectors are parallel to reference
    # axis_norms = np.linalg.norm(axes, axis=1) # (n, )
    # near_zero = axis_norms < 1e-6
    # axes[near_zero] = np.array([0.0, 0.0, 1.0])  # Arbitrary perpendicular axis
    # axis_norms[near_zero] = 1.0
    
    # # Normalize rotation axes
    # axes = axes / axis_norms[:, np.newaxis] # (n, 3)  / (n, 1)
    
    # # Compute rotation vectors
    # rotvecs = angles[:, np.newaxis] * axes  # (n, 1) * (n, 3)
    
    # # Convert to quaternion angles
    # quaternions = axis_angle_to_quaternion(rotvecs)
    
    # Handle vectors aligned with reference (return zero Euler angles)
    # aligned = np.allclose(normalized_vectors, reference, atol=1e-6)
    # quaternions[aligned] = torch.tensor([1.0, 0., 0., 0.])
    
    return quaternions


def vectors_to_euler(vectors):
    """
    Convert a batch of 3D vectors to Euler angles (xyz convention).
    
    Parameters:
    vectors : np.ndarray, shape (n, 3)
        Array of n 3D vectors.
    
    Returns:
    euler_angles : np.ndarray, shape (n, 3)
        Array of Euler angles in radians for each input vector.
    """
    # Ensure input is a numpy array with shape (n, 3)
    vectors = np.asarray(vectors)
    if vectors.ndim != 2 or vectors.shape[1] != 3:
        raise ValueError("Input must be an array of shape (n, 3)")
    
    # Normalize the vectors
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) # (n, 3) -> (n, 1)
    if np.any(norms == 0):
        raise ValueError("Input contains zero vectors")
    normalized_vectors = vectors / norms
    
    # Reference vector
    reference = np.array([1.0, 0.0, 0.0])
    
    # Compute rotation axes and angles
    axes = np.cross(reference, normalized_vectors)
    dots = np.dot(normalized_vectors, reference)
    angles = np.arccos(dots)
    
    # Handle cases where vectors are parallel to reference
    axis_norms = np.linalg.norm(axes, axis=1) # (n, )
    near_zero = axis_norms < 1e-6
    axes[near_zero] = np.array([0.0, 0.0, 1.0])  # Arbitrary perpendicular axis
    axis_norms[near_zero] = 1.0
    
    # Normalize rotation axes
    axes = axes / axis_norms[:, np.newaxis] # (n, 3)  / (n, 1)
    
    # Compute rotation vectors
    rotvecs = angles[:, np.newaxis] * axes  # (n, 1) * (n, 3)
    
    # Convert to Euler angles
    rotation = R.from_rotvec(rotvecs)
    euler_angles = rotation.as_euler('xyz')
    
    # Handle vectors aligned with reference (return zero Euler angles)
    aligned = np.allclose(normalized_vectors, reference, atol=1e-6)
    euler_angles[aligned] = 0.0
    
    return euler_angles


def quaternion_to_axis_angle(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as quaternions to axis/angle.

    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Rotations given as a vector in axis angle form, as a tensor
            of shape (..., 3), where the magnitude is the angle
            turned anticlockwise in radians around the vector's
            direction.
    """
    norms = torch.norm(quaternions[..., 1:], p=2, dim=-1, keepdim=True)
    half_angles = torch.atan2(norms, quaternions[..., :1])
    sin_half_angles_over_angles = 0.5 * torch.sinc(half_angles / torch.pi)
    # angles/2 are between [-pi/2, pi/2], thus sin_half_angles_over_angles
    # can't be zero
    return quaternions[..., 1:] / sin_half_angles_over_angles

def quaternion_to_matrix(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as quaternions to rotation matrices.

    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    r, i, j, k = torch.unbind(quaternions, -1)
    # pyre-fixme[58]: `/` is not supported for operand types `float` and `Tensor`.
    two_s = 2.0 / (quaternions * quaternions).sum(-1)

    o = torch.stack(
        (
            1 - two_s * (j * j + k * k),
            two_s * (i * j - k * r),
            two_s * (i * k + j * r),
            two_s * (i * j + k * r),
            1 - two_s * (i * i + k * k),
            two_s * (j * k - i * r),
            two_s * (i * k - j * r),
            two_s * (j * k + i * r),
            1 - two_s * (i * i + j * j),
        ),
        -1,
    )
    return o.reshape(quaternions.shape[:-1] + (3, 3))

def matrix_to_euler_angles(matrix: torch.Tensor, convention: str) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to Euler angles in radians.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).
        convention: Convention string of three uppercase letters.

    Returns:
        Euler angles in radians as tensor of shape (..., 3).
    """
    if len(convention) != 3:
        raise ValueError("Convention must have 3 letters.")
    if convention[1] in (convention[0], convention[2]):
        raise ValueError(f"Invalid convention {convention}.")
    for letter in convention:
        if letter not in ("X", "Y", "Z"):
            raise ValueError(f"Invalid letter {letter} in convention string.")
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")
    i0 = _index_from_letter(convention[0])
    i2 = _index_from_letter(convention[2])
    tait_bryan = i0 != i2
    if tait_bryan:
        central_angle = torch.asin(
            matrix[..., i0, i2] * (-1.0 if i0 - i2 in [-1, 2] else 1.0)
        )
    else:
        central_angle = torch.acos(matrix[..., i0, i0])

    o = (
        _angle_from_tan(
            convention[0], convention[1], matrix[..., i2], False, tait_bryan
        ),
        central_angle,
        _angle_from_tan(
            convention[2], convention[1], matrix[..., i0, :], True, tait_bryan
        ),
    )
    return torch.stack(o, -1)

def _index_from_letter(letter: str) -> int:
    if letter == "X":
        return 0
    if letter == "Y":
        return 1
    if letter == "Z":
        return 2
    raise ValueError("letter must be either X, Y or Z.")

def _angle_from_tan(
    axis: str, other_axis: str, data, horizontal: bool, tait_bryan: bool
) -> torch.Tensor:
    """
    Extract the first or third Euler angle from the two members of
    the matrix which are positive constant times its sine and cosine.

    Args:
        axis: Axis label "X" or "Y or "Z" for the angle we are finding.
        other_axis: Axis label "X" or "Y or "Z" for the middle axis in the
            convention.
        data: Rotation matrices as tensor of shape (..., 3, 3).
        horizontal: Whether we are looking for the angle for the third axis,
            which means the relevant entries are in the same row of the
            rotation matrix. If not, they are in the same column.
        tait_bryan: Whether the first and third axes in the convention differ.

    Returns:
        Euler Angles in radians for each matrix in data as a tensor
        of shape (...).
    """

    i1, i2 = {"X": (2, 1), "Y": (0, 2), "Z": (1, 0)}[axis]
    if horizontal:
        i2, i1 = i1, i2
    even = (axis + other_axis) in ["XY", "YZ", "ZX"]
    if horizontal == even:
        return torch.atan2(data[..., i1], data[..., i2])
    if tait_bryan:
        return torch.atan2(-data[..., i2], data[..., i1])
    return torch.atan2(data[..., i2], -data[..., i1])

def matrix_to_axis_angle(matrix: torch.Tensor, fast: bool = False) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to axis/angle.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).
        fast: Whether to use the new faster implementation (based on the
            Rodrigues formula) instead of the original implementation (which
            first converted to a quaternion and then back to a rotation matrix).

    Returns:
        Rotations given as a vector in axis angle form, as a tensor
            of shape (..., 3), where the magnitude is the angle
            turned anticlockwise in radians around the vector's
            direction.

    """
    if not fast:
        return quaternion_to_axis_angle(matrix_to_quaternion(matrix))

    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")

    omegas = torch.stack(
        [
            matrix[..., 2, 1] - matrix[..., 1, 2],
            matrix[..., 0, 2] - matrix[..., 2, 0],
            matrix[..., 1, 0] - matrix[..., 0, 1],
        ],
        dim=-1,
    )
    norms = torch.norm(omegas, p=2, dim=-1, keepdim=True)
    traces = torch.diagonal(matrix, dim1=-2, dim2=-1).sum(-1).unsqueeze(-1)
    angles = torch.atan2(norms, traces - 1)

    zeros = torch.zeros(3, dtype=matrix.dtype, device=matrix.device)
    omegas = torch.where(torch.isclose(angles, torch.zeros_like(angles)), zeros, omegas)

    near_pi = angles.isclose(angles.new_full((1,), torch.pi)).squeeze(-1)

    axis_angles = torch.empty_like(omegas)
    axis_angles[~near_pi] = (
        0.5 * omegas[~near_pi] / torch.sinc(angles[~near_pi] / torch.pi)
    )

    # this derives from: nnT = (R + 1) / 2
    n = 0.5 * (
        matrix[near_pi][..., 0, :]
        + torch.eye(1, 3, dtype=matrix.dtype, device=matrix.device)
    )
    axis_angles[near_pi] = angles[near_pi] * n / torch.norm(n)

    return axis_angles

# example_x, angle_x = [1, 0, 0], [0, 0, 0]
# example_y, angle_y = [0, 1, 0], [0, 0, 90.]
# example_z, angle_z = [0, 0, 1], [0, -90, 0.]
# example_45, angle_45 = [1, 1, 0], [0, 0, 45.]
# example_y45, angle_y45 = [1, 0, 1], [0, -45., 0]
# custom_example = [-0.54509804, -0.70980392, -0.43529412]
# # tensor([-0.2261, -0.4527,  0.9159], dtype=torch.float64)
# # tensor([-0.8742,  0.4527, -2.2257], dtype=torch.float64)



# examples = np.array([example_x, example_y, example_z, example_45, example_y45, custom_example, -np.array(custom_example)])
# eulers = vectors_to_euler(examples)
# rot_matrix = euler_angles_to_matrix(torch.tensor(eulers), convention="XYZ")
# references = torch.tile(torch.tensor([1.0, 0, 0], dtype=torch.double), (examples.shape[0], 1) )
# #torch.dot(rot_matrix,)
# # torch.matmul(rot_matrix, references.unsqueeze(-1)).squeeze(-1)

# print(vectors_to_quaternion(torch.tensor(examples)))
# print(vectors_to_quaternion(torch.tensor(examples)))
# values = vectors_to_quaternion(torch.tensor(examples))

In [6]:

def pixel_to_world(x, y, depth, cam_pos, cam_lookat, fov, img_width, img_height, world_up):
    """
    Convert a batch of pixel coordinates and depths to world coordinates.

    Args:
        x: array of pixel x-coordinates (horizontal) [N]
        y: array of pixel y-coordinates (vertical) [N]
        depth: array of depth values at those pixels [N]
        cam_pos: camera position (x, y, z)
        cam_lookat: point camera is looking at (x, y, z)
        fov: field of view in degrees
        img_width: width of the image in pixels
        img_height: height of the image in pixels
        world_up: world up vector (x, y, z)

    Returns:
        world_coords: array of (x, y, z) in world coordinates [N, 3]
    """
    # Convert inputs to numpy arrays
    cam_pos = np.array(cam_pos)
    cam_lookat = np.array(cam_lookat)
    world_up = np.array(world_up)
    x = np.array(x)
    y = np.array(y)
    depth = np.array(depth)

    # Calculate camera direction vector
    cam_dir = cam_lookat - cam_pos
    cam_dir = cam_dir / np.linalg.norm(cam_dir)

    # Calculate camera up vector
    cam_right = np.cross(cam_dir, world_up)
    cam_right = cam_right / np.linalg.norm(cam_right)
    cam_up = np.cross(cam_right, cam_dir)

    # Convert FOV to radians
    fov_rad = np.deg2rad(fov)

    # Calculate direction vector in camera space
    aspect_ratio = img_width / img_height
    tan_half_fov = np.tan(fov_rad / 2.0)

    # Calculate pixel coordinates in normalized device coordinates (-1 to 1)
    ndc_x = (2.0 * x / (img_width - 1)) - 1.0
    ndc_y = 1.0 - (2.0 * y / (img_height - 1))  # Flip y-axis

    # Calculate camera space coordinates
    cam_space_x = ndc_x * tan_half_fov * aspect_ratio
    cam_space_y = ndc_y * tan_half_fov
    cam_space_z = -1.0 * np.ones_like(cam_space_x)  # Negative z is forward

    # Create direction vectors in camera space
    cam_space_dir = np.stack([cam_space_x, cam_space_y, cam_space_z], axis=-1)
    cam_space_dir = cam_space_dir / np.linalg.norm(cam_space_dir, axis=-1, keepdims=True)

    # Convert to world space direction
    world_dir = (cam_right * cam_space_dir[..., 0:1] +
                 cam_up * cam_space_dir[..., 1:2] +
                 -cam_dir * cam_space_dir[..., 2:3])
    world_dir = world_dir / np.linalg.norm(world_dir, axis=-1, keepdims=True)

    # Calculate world positions
    world_pos = cam_pos + world_dir * depth[..., None]

    return world_pos

In [7]:
def check_bottle_contact():
    contacts = spot_gripper.get_contacts(with_entity=plastic_bottle)
    
    if not contacts or 'position' not in contacts:
        return torch.zeros(scene.n_envs)
    
    num_envs, num_contacts, _ = contacts['position'].shape

    link_names = [l.name for l in scene.rigid_solver.links]
    floor_link_idx = link_names.index("bottle_stl_baselink") if "bottle_stl_baselink" in link_names else -1
    gripper_links = ["arm_link_wr1", "arm_link_fngr"]

    if floor_link_idx == -1:
        raise ValueError("No bottle found on links")

    had_contact = torch.zeros( (num_envs, ), dtype=bool )

    for n in range(num_envs):
        for i in range(num_contacts):
            if (not contacts['valid_mask'][n, i]):
                continue

            link_a_idx = contacts['link_a'][n, i]
            link_b_idx = contacts['link_b'][n, i]
            link_a_name = link_names[link_a_idx]
            link_b_name = link_names[link_b_idx]
            
            if link_b_name in gripper_links or link_a_name in gripper_links:
                had_contact[n] = True

    return had_contact

def check_floor_contact():
    contacts = spot_gripper.get_contacts(with_entity=plane)
    
    if not contacts or 'position' not in contacts:
        return torch.zeros(scene.n_envs)
    
    num_envs, num_contacts, _ = contacts['position'].shape

    link_names = [l.name for l in scene.rigid_solver.links]
    floor_link_idx = link_names.index("plane_baselink") if "plane_baselink" in link_names else -1

    if floor_link_idx == -1:
        raise ValueError("No floor found on links")

    had_contact = torch.zeros( (num_envs, ), dtype=bool )

    for n in range(num_envs):
        for i in range(num_contacts):
            if (not contacts['valid_mask'][n, i]):
                continue

            link_a_idx = contacts['link_a'][n, i]
            link_b_idx = contacts['link_b'][n, i]
            link_a_name = link_names[link_a_idx]
            link_b_name = link_names[link_b_idx]

    return had_contact


def perform_grasp(num_steps=10, pause_steps=10, max_gripper_pos=torch.pi/2):

    current_qpos = spot_gripper.get_dofs_position()
    start_gripper_pos = current_qpos[:, finger_dof]
    gripper_step_size = (max_gripper_pos - start_gripper_pos) / num_steps
    
    for i in range(num_steps + 1):
        current_gripper_pos = start_gripper_pos + i * gripper_step_size
        target_qpos = current_qpos.clone()
        target_qpos[:, finger_dof] = current_gripper_pos
        spot_gripper.control_dofs_position(target_qpos)
        
        # Run simulation steps to stabilize and check contact
        for _ in range(pause_steps):
            scene.step()
    
    for _ in range(10):
        scene.step()
    

In [8]:
import torch 
import time
def quaternion_from_euler(angles, seq='XYZ'):
    rot_matrix = euler_angles_to_matrix(angles, convention=seq.upper())
    quat_wxyz = matrix_to_quaternion(rot_matrix)  # Shape: (4,), [w, x, y, z]
    return quat_wxyz

def rotate_90(quaternion):
    converted_matrix_orientation = quaternion_to_matrix(quaternion)
    converted_euler_orientation = matrix_to_euler_angles(converted_matrix_orientation, convention="ZYX")
    converted_euler_orientation[:,2] = np.pi/2
    converted_matrix_from_euler = euler_angles_to_matrix(converted_euler_orientation, convention="ZYX")
    converted_quat_from_axis_angle = matrix_to_quaternion(converted_matrix_from_euler)
    spot_gripper.set_quat(converted_quat_from_axis_angle)

def approach(vector_normal, grasp_point, k_dist = 0.35):
    spot_gripper.set_pos((k_dist) * vector_normal+grasp_point)  


In [9]:
# Output directory
dataset_dir = Path("../water_bottle_dataset")
dataset_dir.mkdir(parents=True, exist_ok=True)

# Verification directory
image_dir = Path("../img_verification_dataset")
image_dir.mkdir(parents=True, exist_ok=True)

# Number of samples
num_samples = 1000

# Up direction reference
up = [0, 0, 1]

In [10]:
''' Randomizing View '''
#randomize camera position
rand_cam_x = 0.5 #random.uniform(-0.5, 0.5)
rand_cam_y = 0.5 # random.uniform(-0.5, 0.5)
rand_cam_z = 0.5 # random.uniform(0.16, 0.6)

# randomize camera lookat
rand_lookat_x = random.uniform(-0.03, 0.03)
rand_lookat_y = random.uniform(-0.03, 0.03)
rand_lookat_z = random.uniform(0, 0.09)   

# Camera common parameters
cam_lookat = [rand_lookat_x, rand_lookat_y, rand_lookat_z]
cam_pos = [rand_cam_x,rand_cam_y,rand_cam_z]
img_width, img_height, fov = 480, 240, 30

i = 0

''' View setup'''
view_id = f"view_{i}"
view_dir = dataset_dir / view_id
img_view_dir = image_dir / view_id
view_dir.mkdir(parents=True, exist_ok=True)
img_view_dir.mkdir(parents=True, exist_ok=True)

# change camera position
cam.set_pose(
    pos = (rand_cam_x,rand_cam_y,rand_cam_z),
    lookat= (rand_lookat_x, rand_lookat_y, rand_lookat_z)
    )

# Get RGB-D data
for i in range(5):
    scene.step()
    render_output = cam.render(rgb=True, depth=True, segmentation=True, normal=True, colorize_seg=False, )

rgb, depth, seg, normal_map = render_output
cylinder_pixel_coords = np.where(np.array(seg) == 1)
cylinder_pixel_coords=np.transpose(cylinder_pixel_coords)

# Save data
np.save(view_dir / "rgb.npy", rgb)
np.save(view_dir / "depth.npy", depth)
np.save(view_dir / "segmentation.npy", seg)
np.save(view_dir / "normal.npy", normal_map)

# Save PNGs for visual verification
cv2.imwrite(str(img_view_dir / "rgb.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))


True

In [11]:
# TODO: select different i for each environment
assert scene.n_envs <= len(cylinder_pixel_coords)
print(len(cylinder_pixel_coords))
random_idx = np.linspace(start=0, stop=len(cylinder_pixel_coords)-1, num=scene.n_envs, dtype=int)
random_y, random_x = cylinder_pixel_coords[random_idx, :].T
specific_point_depth = depth[random_y, random_x]  # (5, )
specific_point_normal = normal_map[random_y, random_x]
vector_normal = 2 * (specific_point_normal - 127.5) / 255

grasp_point = pixel_to_world(random_x, random_y, specific_point_depth, cam_pos, cam_lookat, fov, img_width, img_height, up)
quaternions = vectors_to_quaternion(-torch.tensor(vector_normal))
position = grasp_point + 0.5* vector_normal.copy()

spot_gripper.set_pos(position)
spot_gripper.set_quat(quaternions)

rotate_90(quaternions)

approach(vector_normal, grasp_point, k_dist=0.18)
for _ in range(2):
    scene.step()


/tmp/ipykernel_676014/2710013593.py:50: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/Cross.cpp:62.)
  axes = torch.cross(reference.expand_as(normalized_vectors), normalized_vectors)
[Genesis] [16:45:26] [WARNING] Base link is fixed. Overriding base link pose.


3031


[Genesis] [16:45:26] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [16:45:27] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [16:45:27] [WARNING] Base link is fixed. Overriding base link pose.


In [12]:
perform_grasp()

for _ in range(2):
    scene.step()
bottle_contacts = check_bottle_contact()
floor_contacts = check_floor_contact()
final_contacts = torch.logical_and(bottle_contacts, ~floor_contacts)
final_contacts

tensor([True, True, True, True, True])

In [13]:
final_contacts =  final_contacts.cpu().numpy().tolist()

# Store metadata
data_metadata = []
data_metadata.append({
    "view_id": view_id,
    "camera_pos": cam_pos,
    "camera_lookat": cam_lookat,
    "final_contacts": final_contacts,
    "random_y":random_y.tolist(), 
    "random_x":random_x.tolist(), 
})

# Save metadata
with open( view_dir / "metadata.json", "w") as f:
    json.dump(data_metadata, f, indent=4)

# with open(view_dir / "grasp_success.txt", "w") as f:
#     f.write(str(final_contacts))        